In [1]:
import hyperspy.api as hs
import pyxem as pxm
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import matplotlib.patheffects as patheffects
from matplotlib.colors import LinearSegmentedColormap,TwoSlopeNorm,SymLogNorm
from scipy import ndimage

%matplotlib qt5

# Define functions

In [2]:
# Find the time of the first frame in M1
def convert_filetime(ft):
    return datetime(1601, 1, 1) + timedelta(microseconds=ft / 10)




# Open the files


def open_movie(path):
    "Open the movie, stack all TEM images lazy and extract the acquisition times and temperature log"

    root = Path(path)
    files = [
        f for f in root.rglob("*")
        if (
            f.is_file()
            and not f.name.startswith("._")  
            and f.suffix.lower() == ".dm4"
        )
    ]


    # Sort files on the aqcuisition time
    data = []
    for f in files[1:]: #skip wildfire log files
        try:
            s_temporary = hs.load(f, lazy=True,load_original_metadata=True,signal_type=None)
            ft = s_temporary.original_metadata.ImageList.TagGroup0.ImageTags.DataBar.Acquisition_Time_OS
            t = convert_filetime(ft)
            data.append((f, t))
            del s_temporary
        except Exception as e:
            print(f"Corrupted file skipped: {f.name} - Error: {type(e).__name__}")

    data_sorted = sorted(data, key=lambda x: x[1])
    files_sorted = [d[0] for d in data_sorted]
    timestamps_sorted = [d[1] for d in data_sorted] #list of daytime objects


    # Open signals lazy as a stack
    signals = [hs.load(f, lazy=True) for f in files_sorted]
    s = hs.stack(signals)


    # Open log data 
    global Wildfire_Holder, log, time_log, setpoint, power, temperature
    Wildfire_Holder = hs.load(files[0])
    log = Wildfire_Holder[0]
    temperature = Wildfire_Holder[0].data
    time_log = Wildfire_Holder[0].axes_manager[0].axis # time of temperature readout
    setpoint = Wildfire_Holder[1]
    power = Wildfire_Holder[2]


    # Time of the first frame in M1
    global first_image
    first_image = hs.load('/Volumes/PNYRP60PSSD/W2/M4/Hour_00/Minute_00/Second_00/M4_Hour_00_Minute_00_Second_00_Frame_0000.dm4')
    fileTime = first_image.original_metadata.ImageList.TagGroup0.ImageTags.DataBar.Acquisition_Time_OS

    
    #t0_global = convert_filetime(fileTime)
    t0 = timestamps_sorted[0]
    #print(t0.timestamp())
    start_time = 1773959985.642008 #time of first frame in M1
    seconds_since_t0 = [t.timestamp() - start_time for t in timestamps_sorted]
    

    # Make the time grid from the wildfier log and the images exisit in the same coordinate system
    time_log_abs = [t0 + timedelta(seconds=t) for t in time_log]
    time_log_rel = np.array([
        (t - t0).total_seconds() for t in time_log_abs])


    # Interpolate to find temperature of image acqusition
    time_aquisition = np.array([(t - t0).total_seconds() for t in timestamps_sorted]) #time of acquisitions
    T_aquisition = np.interp(time_aquisition, time_log_rel, temperature) #temperature interpolated

    return s, time_aquisition, T_aquisition, time_log_rel, seconds_since_t0


# Description
'''Extracts precise acquisition timestamps from individual TEM images,
sorts them chronologically, and stacks them into a time-ordered dataset.
It then aligns these image times with the DENS temperature log using a common reference time
and interpolates the temperature so each frame is assigned its corresponding temperature.'''

'Extracts precise acquisition timestamps from individual TEM images,\nsorts them chronologically, and stacks them into a time-ordered dataset.\nIt then aligns these image times with the DENS temperature log using a common reference time\nand interpolates the temperature so each frame is assigned its corresponding temperature.'

In [5]:
p = '/Volumes/PNYRP60PSSD/W2/M4'
s, time_aquisition, T_aquisition, time_log_rel = open_movie(p)

WARNING | Hyperspy | Axis calibration mismatch detected along axis 0. The calibration of signal 0 along this axis will be applied to all signals after stacking. (hyperspy.misc.signal_tools:109)
WARNING | Hyperspy | Axis calibration mismatch detected along axis 1. The calibration of signal 0 along this axis will be applied to all signals after stacking. (hyperspy.misc.signal_tools:109)


ValueError: too many values to unpack (expected 4)

In [8]:
s.axes_manager

Navigation axis name,size,index,offset,scale,units
stack_element,332,0,0.0,1.0,
Signal axis name,size,,offset,scale,units
x,4096,,-0.0,0.00686672143638134,1/nm
y,4096,,-0.0,0.00686672143638134,1/nm


In [164]:
first_image.original_metadata

├── BackgroundColor = (-2571, -2571, -1286)
├── DocumentObjectList
│   └── TagGroup0
│       ├── AnnotationGroupList
│       ├── AnnotationType = 20
│       ├── BackgroundColor = (-1, -1, -1)
│       ├── BackgroundMode = 2
│       ├── FillMode = 1
│       ├── ForegroundColor = (-1, 0, 0)
│       ├── HasBackground = 0
│       ├── ImageDisplayInfo
│       │   ├── BrightColor = (-1, -1, -1)
│       │   ├── Brightness = 0.5
│       │   ├── CLUT <list>
│       │   │   ╠══ [0] = (0, 0, 0)
│       │   │   ╠══ [1] = (257, 257, 257)
│       │   │   ╠══ [10] = (2570, 2570, 2570)
│       │   │   ╠══ [100] = (25700, 25700, 25700)
│       │   │   ╠══ [101] = (25957, 25957, 25957)
│       │   │   ╠══ [102] = (26214, 26214, 26214)
│       │   │   ╠══ [103] = (26471, 26471, 26471)
│       │   │   ╠══ [104] = (26728, 26728, 26728)
│       │   │   ╠══ [105] = (26985, 26985, 26985)
│       │   │   ╠══ [106] = (27242, 27242, 27242)
│       │   │   ╠══ [107] = (27499, 27499, 27499)
│       │   │   ╠══ [108] = (27756, 27756, 27756)
│       │   │   ╠══ [109] = (28013, 28013, 28013)
│       │   │   ╠══ [11] = (2827, 2827, 2827)
│       │   │   ╠══ [110] = (28270, 28270, 28270)
│       │   │   ╠══ [111] = (28527, 28527, 28527)
│       │   │   ╠══ [112] = (28784, 28784, 28784)
│       │   │   ╠══ [113] = (29041, 29041, 29041)
│       │   │   ╠══ [114] = (29298, 29298, 29298)
│       │   │   ╠══ [115] = (29555, 29555, 29555)
│       │   │   ╠══ [116] = (29812, 29812, 29812)
│       │   │   ╠══ [117] = (30069, 30069, 30069)
│       │   │   ╠══ [118] = (30326, 30326, 30326)
│       │   │   ╠══ [119] = (30583, 30583, 30583)
│       │   │   ╠══ [12] = (3084, 3084, 3084)
│       │   │   ╠══ [120] = (30840, 30840, 30840)
│       │   │   ╠══ [121] = (31097, 31097, 31097)
│       │   │   ╠══ [122] = (31354, 31354, 31354)
│       │   │   ╠══ [123] = (31611, 31611, 31611)
│       │   │   ╠══ [124] = (31868, 31868, 31868)
│       │   │   ╠══ [125] = (32125, 32125, 32125)
│       │   │   ╠══ [126] = (32382, 32382, 32382)
│       │   │   ╠══ [127] = (32639, 32639, 32639)
│       │   │   ╠══ [128] = (-32640, -32640, -32640)
│       │   │   ╠══ [129] = (-32383, -32383, -32383)
│       │   │   ╠══ [13] = (3341, 3341, 3341)
│       │   │   ╠══ [130] = (-32126, -32126, -32126)
│       │   │   ╠══ [131] = (-31869, -31869, -31869)
│       │   │   ╠══ [132] = (-31612, -31612, -31612)
│       │   │   ╠══ [133] = (-31355, -31355, -31355)
│       │   │   ╠══ [134] = (-31098, -31098, -31098)
│       │   │   ╠══ [135] = (-30841, -30841, -30841)
│       │   │   ╠══ [136] = (-30584, -30584, -30584)
│       │   │   ╠══ [137] = (-30327, -30327, -30327)
│       │   │   ╠══ [138] = (-30070, -30070, -30070)
│       │   │   ╠══ [139] = (-29813, -29813, -29813)
│       │   │   ╠══ [14] = (3598, 3598, 3598)
│       │   │   ╠══ [140] = (-29556, -29556, -29556)
│       │   │   ╠══ [141] = (-29299, -29299, -29299)
│       │   │   ╠══ [142] = (-29042, -29042, -29042)
│       │   │   ╠══ [143] = (-28785, -28785, -28785)
│       │   │   ╠══ [144] = (-28528, -28528, -28528)
│       │   │   ╠══ [145] = (-28271, -28271, -28271)
│       │   │   ╠══ [146] = (-28014, -28014, -28014)
│       │   │   ╠══ [147] = (-27757, -27757, -27757)
│       │   │   ╠══ [148] = (-27500, -27500, -27500)
│       │   │   ╠══ [149] = (-27243, -27243, -27243)
│       │   │   ╠══ [15] = (3855, 3855, 3855)
│       │   │   ╠══ [150] = (-26986, -26986, -26986)
│       │   │   ╠══ [151] = (-26729, -26729, -26729)
│       │   │   ╠══ [152] = (-26472, -26472, -26472)
│       │   │   ╠══ [153] = (-26215, -26215, -26215)
│       │   │   ╠══ [154] = (-25958, -25958, -25958)
│       │   │   ╠══ [155] = (-25701, -25701, -25701)
│       │   │   ╠══ [156] = (-25444, -25444, -25444)
│       │   │   ╠══ [157] = (-25187, -25187, -25187)
│       │   │   ╠══ [158] = (-24930, -24930, -24930)
│       │   │   ╠══ [159] = (-24673, -24673, -24673)
│       │   │   ╠══ [16] = (4112, 4112, 4112)
│       │   │   ╠══ [160] = (-24416, -24416, -24416)

# Inspect time and temperature signals

In [7]:
measured_T = np.array(temperature)
set_T = np.array(setpoint)

time_measured = time_log
time_set = Wildfire_Holder[1].axes_manager[0].axis
plt.plot(time_measured,measured_T,label='Measured Temperature')
plt.plot(time_set,set_T,label='Set Temperature')
plt.xlabel('Time (s)')
plt.ylabel('Temperature (C)')
plt.legend()
plt.show()

rmse = np.sqrt(np.mean((set_T - measured_T)**2))
print(f' The root mean square of the measured vs setpoint tempereature is: {round(rmse,4)} C')

 The root mean square of the measured vs setpoint tempereature is: 0.1148 C


In [7]:
plt.close()

In [8]:
log.axes_manager

Signal axis name,size,,offset,scale,units
x,385,,-0.23252970965286224,0.33105090260505676,time since 2026-03-18 23:13:14.072 (seconds)


# Inspect images 

In [26]:
s.axes_manager

Navigation axis name,size,index,offset,scale,units
stack_element,26,0,0.0,1.0,
Signal axis name,size,,offset,scale,units
x,4096,,-0.0,0.00686672143638134,1/nm
y,4096,,-0.0,0.00686672143638134,1/nm


In [7]:
first_image.plot(vmin=25,vmax=2400,cmap='inferno')

In [23]:
s.inav[0].plot(vmin=0,vmax=2500)

  0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
first_im = first_image.copy()
first_im = ndimage.rotate(first_im, 0)
f = first_im[200:3896, 400:4096]

In [12]:
plt.imshow(f, cmap='inferno',vmax=2500)

# Calculating differences

In [7]:
# Set calibration of diffraction pattern
first_image.plot(vmin=0,vmax=10000)
first_image.calibrate() #distance from (0000) to (-3300) is 5.794 nm^-1 (5.794*2 = 11.588 nm^-1)

In [8]:
# Matplotlib settings

# Define colors: negative → center → positive
colors = [
    (0.0, 0.0, 1.0),  # blue
    (0.7, 0.7, 0.7),  # black (zero)
    (1.0, 0.0, 0.0)   # red
]

costom_cmap = LinearSegmentedColormap.from_list(
    "costom_cmap",
    colors,
    N=256
)

norm = TwoSlopeNorm(
    vmin=-400,
    vcenter=0,
    vmax=400
)

# norm = SymLogNorm(
#     vmin=-500,
#     linthresh=10,
#     vmax=500
# )

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Lucida Grande']
fontprops = fm.FontProperties(size=16)

# sets scale and extent for plotting

# first_im = first_image.copy()
# first_im = ndimage.rotate(first_im, 0)
# f = first_im[200:3896, 400:4096]

# scale = first_image.axes_manager[0].scale
# ext = [0,scale*f.shape[1],0,scale*f.shape[0]]



scale = first_image.axes_manager[0].scale
ext = [0,scale*first_image.data.shape[1],0,scale*first_image.data.shape[0]]

In [ ]:
plt.ioff() # turn off interactive mode to prevent figures from popping up while running the loop

PATH = '/Users/jonasfsimonsen/Documents/NTNU/5klasse/Masteroppgave/DataHandling/ETEM/W2/'
movie_paths = [
    '/Volumes/PNYRP60PSSD/W2/M4',
    '/Volumes/PNYRP60PSSD/W2/M7',
    '/Volumes/PNYRP60PSSD/W2/M8',
    '/Volumes/PNYRP60PSSD/W2/M9',
    '/Volumes/PNYRP60PSSD/W2/M10',
    '/Volumes/PNYRP60PSSD/W2/M11',
    '/Volumes/PNYRP60PSSD/W2/M12',
    '/Volumes/PNYRP60PSSD/W2/M13',
    '/Volumes/PNYRP60PSSD/W2/M14',
    '/Volumes/PNYRP60PSSD/W2/M15'
    ]

# movie_paths = [
#     '/Volumes/PNYRP60PSSD/W3/M2',
#     '/Volumes/PNYRP60PSSD/W3/M15',
#     '/Volumes/PNYRP60PSSD/W3/M18',
#     '/Volumes/PNYRP60PSSD/W3/M21',
#     '/Volumes/PNYRP60PSSD/W3/M22',    
#     '/Volumes/PNYRP60PSSD/W3/M24',
#     '/Volumes/PNYRP60PSSD/W3/M25',
#     '/Volumes/PNYRP60PSSD/W3/M26',
#     '/Volumes/PNYRP60PSSD/W3/M27',
#     '/Volumes/PNYRP60PSSD/W3/M28'
  
#     ]





for i, path in enumerate(movie_paths):
    s, time_aquisition, T_aquisition, time_log_rel, fileTime = open_movie(path)

    print(f'Cycle {i+1}:\nTime start: {round(fileTime[0], 0)}, Time end: {round(fileTime[-1], 0)}')
    print(f'Time start: {round(fileTime[0]/60, 1)} min, Time end: {round(fileTime[-1]/60, 1)} min\n')
    img1 = s.inav[0]
    img2 = s.inav[-1]

    # Diffrerence between first and last frame in the cycle
    img1 = img1.data.compute()
    img2 = img2.data.compute()


    # img1 = ndimage.rotate(img1, -3)
    # img1 = img1[200:3896, 400:4096]
    # img2 = ndimage.rotate(img2, -3)
    # img2 = img2[200:3896, 400:4096]
    
    difference = hs.signals.Signal2D(img2 - img1)

    # diff_value = np.sum(np.abs(difference.data)) # Is this a good way to quantify the difference between the first and last frame? 
    # print(f"Difference value between T_start and T_end for Cycle {i+1}: {diff_value}")


    # difference between the first frame of the current cycle and the last frame of the previous cycle
    if i == 0:
        img_previous_cycle = img2 # Dummy variable for the first round to make the code work, this will be overwritten in the next round
    difference_to_previous_cycle = hs.signals.Signal2D(img1 - img_previous_cycle)
    img_previous_cycle = img2 # Update the previous cycle image for the next iteration
    

    
    fig, ax = plt.subplots()
    im = ax.imshow(X = img1.data, extent = ext,cmap='inferno',vmin=25,vmax=2400)
    ax.set_xticks([])
    ax.set_yticks([])
    scalebar = AnchoredSizeBar(
                transform=ax.transData,
                size=5,
                label='',
                loc='center',               
                frameon=False,
                color="w",
                size_vertical= 0.6,
                label_top=False,
                fontproperties=fontprops,
                bbox_to_anchor=(0.11, 0.02), # (x, y) in axes coordinates
                bbox_transform=ax.transAxes # interpret the coords in axes space
            )
    ax.add_artist(scalebar)
    filename = PATH+f'Images_used/Cycle{i+1}_start.svg'
    fig.savefig(
            filename, 
            bbox_inches='tight',   # Remove extra whitespace
            pad_inches=0,          # No padding
            dpi=600)
    plt.close(fig)





    fig, ax = plt.subplots()
    im = ax.imshow(X = img2.data, extent = ext,cmap='inferno',vmin=25,vmax=2400)
    ax.set_xticks([])
    ax.set_yticks([])
    scalebar = AnchoredSizeBar(
                transform=ax.transData,
                size=5,
                label='',
                loc='center',               
                frameon=False,
                color="w",
                size_vertical=0.6,
                label_top=False,
                fontproperties=fontprops,
                bbox_to_anchor=(0.11, 0.02), # (x, y) in axes coordinates
                bbox_transform=ax.transAxes # interpret the coords in axes space
            )
    ax.add_artist(scalebar)
    filename = PATH+f'Images_used/Cycle{i+1}_end.svg'
    fig.savefig(
            filename, 
            bbox_inches='tight',   # Remove extra whitespace
            pad_inches=0,          # No padding
            dpi=600)
    plt.close(fig)






    # save images
    images = [difference, difference_to_previous_cycle]
    for j in range(len(images)):
        fig, ax1 = plt.subplots()
        im = ax1.imshow(X = images[j].data, extent = ext,cmap=costom_cmap,norm=norm)
        ax1.set_xticks([])
        ax1.set_yticks([])

        cbar = fig.colorbar(im, ax=ax1)

        # Difference within a cycle
        if j==0:
            # Add title and scalebar
            fig.suptitle(f"Difference in Cycle{i+1}: T2 = {round(T_aquisition[-1], 2)} T1 = {round(T_aquisition[0], 2)}", fontsize=16)
            scalebar = AnchoredSizeBar(
                transform=ax1.transData,
                size=5,
                label='',
                loc='center',               
                frameon=False,
                color="w",
                size_vertical=0.6,
                label_top=False,
                fontproperties=fontprops,
                bbox_to_anchor=(0.11, 0.02), # (x, y) in axes coordinates
                bbox_transform=ax1.transAxes # interpret the coords in axes space
            )
            ax1.add_artist(scalebar)

            # save figure
            filename = PATH+f'DP_Differences_within_cycles/Difference_Cycle{i+1}.svg'
            fig.savefig(
                    filename, 
                    bbox_inches='tight',   # Remove extra whitespace
                    pad_inches=0,          # No padding
                    dpi=600)
        
        # Difference between the end of one cycle and the start of the next cycle
        elif j==1 and i>0:
            # Add title and scalebar
            fig.suptitle(f"Difference between Cycle{i+1} and  Cycle{i}", fontsize=16)
            scalebar = AnchoredSizeBar(
                transform=ax1.transData,
                size=5,
                label='',
                loc='center',               
                frameon=False,
                color="w",
                size_vertical=0.6,
                label_top=False,
                fontproperties=fontprops,
                bbox_to_anchor=(0.11, 0.02), # (x, y) in axes coordinates
                bbox_transform=ax1.transAxes # interpret the coords in axes space
            )
            ax1.add_artist(scalebar)
            # save figure
            filename = PATH+f'DP_Differences_between_cycles/Difference_Cycle{i+1}_and_Cycle{i}.svg'
            fig.savefig(
                    filename, 
                    bbox_inches='tight',   # Remove extra whitespace
                    pad_inches=0,          # No padding
                    dpi=600
        plt.close(fig)
    print(f"Finished processing Cycle {i+1}/{len(movie_paths)}")

WARNING | Hyperspy | Axis calibration mismatch detected along axis 0. The calibration of signal 0 along this axis will be applied to all signals after stacking. (hyperspy.misc.signal_tools:109)
WARNING | Hyperspy | Axis calibration mismatch detected along axis 1. The calibration of signal 0 along this axis will be applied to all signals after stacking. (hyperspy.misc.signal_tools:109)
Cycle 1:
Time start: 0.0, Time end: 37.0
Time start: 0.0 min, Time end: 0.6 min

Finished processing Cycle 1/10
WARNING | Hyperspy | Axis calibration mismatch detected along axis 0. The calibration of signal 0 along this axis will be applied to all signals after stacking. (hyperspy.misc.signal_tools:109)
WARNING | Hyperspy | Axis calibration mismatch detected along axis 1. The calibration of signal 0 along this axis will be applied to all signals after stacking. (hyperspy.misc.signal_tools:109)
Cycle 2:
Time start: 515.0, Time end: 846.0
Time start: 8.6 min, Time end: 14.1 min

Finished processing Cycle 2